<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install deepxde

In [24]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import torch
import pandas as pd

In [25]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [26]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [27]:
data = dde.data.TimePDE(
    geomtime,
    pde,
    [],
    num_domain=114,
    train_distribution="uniform"
)

In [28]:
layer_size = [2] + [20] * 2 + [1]
activation = "tanh"
initializer = "Glorot uniform"
net = dde.nn.FNN(layer_size, activation, initializer)

In [29]:
def output_transform(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

net.apply_output_transform(output_transform)

In [30]:
model = dde.Model(data, net)
model.compile("adam", lr=1e-3)
losshistory, train_state = model.train(iterations=1000, display_every=50)

Compiling model...
'compile' took 0.000444 s

Training model...

Step      Train loss    Test loss     Test metric
0         [9.15e-01]    [9.15e-01]    []  
50        [3.40e-02]    [3.40e-02]    []  
100       [1.12e-02]    [1.12e-02]    []  
150       [8.54e-03]    [8.54e-03]    []  
200       [7.03e-03]    [7.03e-03]    []  
250       [5.74e-03]    [5.74e-03]    []  
300       [4.66e-03]    [4.66e-03]    []  
350       [3.77e-03]    [3.77e-03]    []  
400       [3.05e-03]    [3.05e-03]    []  
450       [2.46e-03]    [2.46e-03]    []  
500       [1.99e-03]    [1.99e-03]    []  
550       [1.60e-03]    [1.60e-03]    []  
600       [1.30e-03]    [1.30e-03]    []  
650       [1.05e-03]    [1.05e-03]    []  
700       [8.51e-04]    [8.51e-04]    []  
750       [6.94e-04]    [6.94e-04]    []  
800       [5.70e-04]    [5.70e-04]    []  
850       [4.73e-04]    [4.73e-04]    []  
900       [3.96e-04]    [3.96e-04]    []  
950       [3.36e-04]    [3.36e-04]    []  
1000      [2.90e-04]    [

In [31]:
points = data.train_x
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, np.pi, np.pi, 0, 0], y=[0, 0, 1, 1, 0], mode="lines", fill="toself", fillcolor="rgba(0, 0, 255, 0.05)"))
fig.add_trace(go.Scatter(x=points[:, 0], y=points[:, 1], mode="markers"))
fig.update_layout(xaxis_title="x", yaxis_title="t", showlegend=False)
fig.show()

In [32]:
domain_x = [0, np.pi]
domain_t = [0, 1]

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
    x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
    t_plot = t_val * np.ones_like(x_plot)
    inputs_plot = np.concatenate([x_plot, t_plot], axis=1)

    p_plot = model.predict(inputs_plot)

    p_exact = np.exp(-t_val) * np.sin(x_plot)
    df_temp = pd.DataFrame({
        'x': x_plot.flatten(),
        'PINN': p_plot.flatten(),
        'Exact': p_exact.flatten(),
        'Time': np.round(t_val, 3)
    })
    plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')

fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type', range_y=[-0.05, 1.05] )
fig.update_layout(xaxis_title='x', yaxis_title='p(x, t)')
fig.show()